# 피드백 1~3: 기여도 분리 · 전체 분포 증류 · 교사 데이터 Oracle

기본 정답 수 N=[200, 2000], 라벨 없는 자료 M=2,000, seed 1~5로 **90회**를 실행합니다. 1~3번을 완전히 끝내려면 모든 계획 회차가 완료되어야 합니다.

| 항목 | 실행 조건 | 목적 |
|---|---|---|
| 1 | fixed_single / fixed_set / meta_single / kampa | 학습한 메타 결합기와 두 후보 손실의 2×2 기여도 |
| 1 보조 | meta_top1_matched | KAMPA와 같은 채택 대상·가중치에서 정답만 top-1로 바꾼 대조군 |
| 2 | soft_selected / soft_all | 전체 9차원 통합 분포를 이용한 증류 |
| 3 | supervised / oracle | 같은 학생에 교사 학습 데이터의 정답을 추가한 특권적 지도학습 비교 |

fixed 분포는 q_k ∝ p_k/(1-p_k), q_others ∝ 1인 비학습 결합 규칙입니다. 기존 논문의 raw-score fixed rule과 별개의 통제 조건입니다. 두 후보는 q_top1<0.9 및 q_top1+q_top2≥0.95일 때 사용합니다.

soft_selected는 KAMPA와 같은 샘플·가중치를 유지하고 손실만 full-q cross entropy(T=1)로 바꿉니다. soft_all은 U 전체 2,000개·균등 가중치의 통합 분포 증류 대조군이므로 채택 범위와 가중치도 달라집니다.

oracle은 현재 교사 TRAIN 16,000행의 실제 클래스를 출처 기록으로 복원합니다. 동일 파일을 제거한 12,477개(원본 4,694개·증강 7,783개, 출처 그룹 4,089개)에서 학생 지도 샘플과의 중복도 다시 제거합니다. 증강 파일 수를 독립적인 새 정답 개수로 세지 않습니다. 교사 VAL/TEST, 학생 VAL/TEST, 숨겨진 U 정답은 추가 학습 라벨로 쓰지 않습니다. 추가 정답 비용이 있는 참고 성능이며 수학적으로 보장되는 상한은 아닙니다.

중단·재실행: 같은 CPU 또는 GPU 환경과 설정으로 위에서부터 다시 실행하면 됩니다. 마지막으로 완전히 저장된 에폭에서 이어집니다. 완료 표식과 산출물 해시가 일치하는 회차는 건너뜁니다. 코드·학습 설정·입력이 달라지면 다른 결과 폴더를 사용해야 합니다.

학습은 최대 80에폭, validation loss 기준 early stopping 10, Adam lr=1e-4, weight decay=1e-4, batch=16입니다. 시험셋은 최적 체크포인트를 고른 후 평가에만 사용합니다. 원본 Mark5.3의 체크포인트·가짜 라벨·예측 파일은 수정하지 않습니다.

새 대조군까지 이 실행 코드로 학습합니다. 기존 논문의 결과와 숫자를 섞어 동일 조건의 새 ablation 표를 만들면 안 됩니다. 결과가 예상과 다르더라도 시험셋을 보고 임계값이나 분할을 바꾸지 마세요.

실행 환경: CPU를 지원합니다. GPU를 사용하면 L4를 우선 사용하고 Blackwell 계열은 기존 프로젝트 정책상 제외합니다. 환경별 결과 폴더가 자동 분리되며 서로 다른 환경의 결과를 합산하지 않습니다.

입력 검증: 원본 FLOAT 음원과 실제 PCM_16 입력의 SHA-256을 별도로 확인합니다. 교사 8개·점수 캐시·전체 입력의 버전이 모두 일치할 때 기존 교사 점수를 재사용하며, 교사 재학습이나 재추론은 필요하지 않습니다.


## 1. Google Drive 연결
드라이브 `NewProject/codes/experiments/Mark5.3/feedback/20261114/notebooks/`의 이 노트북을 Colab에서 열고 '모두 실행'을 누르세요. GPU는 L4를 사용합니다. CPU로도 실행할 수 있습니다.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. 환경과 실행 코드 준비
실행 코드는 같은 폴더의 `KAMPA_추가실험_실행코드.zip`에서 풀고, 결과 집계는 같은 폴더의 `KAMPA_결과집계.py`를 씁니다. 두 파일 중 하나라도 이 노트북에 적힌 SHA-256과 다르면 멈춥니다. Torch 환경 검사는 별도의 Python 프로세스에서 수행합니다. 실행에 시간 제한을 두지 않습니다.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "pandas>=2.0", "scipy>=1.11", "scikit-learn>=1.3", "soundfile>=0.12"], check=True)
# Colab이 제공한 서로 호환되는 torch/torchaudio/torchvision 조합을 유지합니다.
subprocess.run([sys.executable, "-c", "import torch,torchaudio,torchvision; print('torch:',torch.__version__,'torchaudio:',torchaudio.__version__,'torchvision:',torchvision.__version__); print('CUDA available:',torch.cuda.is_available())"], check=True)


In [ ]:
from pathlib import Path
import hashlib, io, json, zipfile
PROJECT_ROOT = Path("/content/drive/MyDrive/NewProject/codes/experiments/Mark5.3")
CODE_ZIP = PROJECT_ROOT / "feedback/20261114/notebooks/KAMPA_추가실험_실행코드.zip"
CODE_SHA256 = "45342d1b0e8cf9f4f8fa946daa6c4e6ef39b4ffbb9225c6f0602ff4281592bf3"
if not CODE_ZIP.exists():
    raise SystemExit(f"실행 코드 묶음이 없습니다: {CODE_ZIP}")
archive = CODE_ZIP.read_bytes()
archive_sha = hashlib.sha256(archive).hexdigest()
if archive_sha != CODE_SHA256:
    raise SystemExit(f"실행 코드 묶음의 SHA-256이 노트북과 다릅니다: {archive_sha[:12]} (기대값 {CODE_SHA256[:12]})")
PACKAGE = Path("/content/kampa_feedback_20261114_v2_45342d1b0e8c")
PACKAGE.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(archive)) as z:
    assert all(not Path(n).is_absolute() and ".." not in Path(n).parts for n in z.namelist())
    z.extractall(PACKAGE)
# 매번 묶음에서 복원하므로 이전 실행 코드가 섞이지 않습니다.
print("실행 코드 준비 완료:", PACKAGE, "SHA-256", CODE_SHA256[:12])
AGGREGATOR_FILE = PROJECT_ROOT / "feedback/20261114/notebooks/KAMPA_결과집계.py"
AGGREGATOR_SHA256 = "6105efac2a7c90bea2d1a6dd54d908b127cb2af0d8123dd98aecdc0ae3824d37"
if not AGGREGATOR_FILE.exists():
    raise SystemExit(f"결과 집계 파일이 없습니다: {AGGREGATOR_FILE}")
aggregator_bytes = AGGREGATOR_FILE.read_bytes()
aggregator_sha = hashlib.sha256(aggregator_bytes).hexdigest()
if aggregator_sha != AGGREGATOR_SHA256:
    raise SystemExit(f"결과 집계 파일의 SHA-256이 노트북과 다릅니다: {aggregator_sha[:12]} (기대값 {AGGREGATOR_SHA256[:12]})")
# 실행 코드 폴더 밖에 둡니다. 실행 코드의 .py 해시가 회차 식별값에 들어가므로 완료 회차가 그대로 인정됩니다.
AGGREGATOR = Path("/content/kampa_aggregate_6105efac2a7c/KAMPA_결과집계.py")
AGGREGATOR.parent.mkdir(parents=True, exist_ok=True)
AGGREGATOR.write_bytes(aggregator_bytes)
print("결과 집계 준비 완료:", AGGREGATOR, "SHA-256", AGGREGATOR_SHA256[:12])


## 3. 실행 범위와 저장 위치
기본은 N=[200, 2000]입니다. 두 라벨 수를 모두 실행하며, 같은 환경과 설정으로 이미 완료한 회차는 검증 후 건너뜁니다.

In [ ]:
probe = subprocess.run([sys.executable, "-c", "import json; from feedback_core import runtime_metadata; print(json.dumps(runtime_metadata(),sort_keys=True))"], cwd=PACKAGE, capture_output=True, text=True, check=True)
RUNTIME = json.loads(probe.stdout)
RUNTIME_ID = hashlib.sha256(json.dumps(RUNTIME,sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_ROOT = PROJECT_ROOT.parent / "KAMPA_feedback_20261114_v2" / RUNTIME_ID / "1_3"
LABEL_BUDGETS = [200, 2000]
TRAIN_SEEDS = [1, 2, 3, 4, 5]
SPLIT_SEEDS = []
CONFIG = dict(root=str(PROJECT_ROOT),output=str(OUTPUT_ROOT),stage="1_3",
              budgets=LABEL_BUDGETS,seeds=TRAIN_SEEDS,split_seeds=SPLIT_SEEDS,
              epochs=80,cache="/content/kampa_feedback_audio_cache_v2_"+RUNTIME_ID)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = OUTPUT_ROOT / "launch_config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG,ensure_ascii=False,indent=2),encoding="utf-8")
(OUTPUT_ROOT/"launch_runtime.json").write_text(json.dumps(RUNTIME,ensure_ascii=False,indent=2),encoding="utf-8")
print("환경:",RUNTIME)
print("결과 저장:",OUTPUT_ROOT)
done = len(list((OUTPUT_ROOT/"runs").glob("*/complete.json")))
print(f"이 결과 폴더에 완료 표식이 있는 회차: {done}개. 해시를 확인한 뒤 건너뜁니다.")
for other in sorted(OUTPUT_ROOT.parent.parent.glob("*/1_3")):
    count = len(list((other/"runs").glob("*/complete.json")))
    if other != OUTPUT_ROOT and count:
        print(f"다른 환경 폴더 {other.parent.name}에 완료 회차 {count}개가 있습니다. 런타임 환경이 달라 이번 실행에서는 이어 쓰지 않습니다.")


## 4. 학습 전 데이터·분할 검사

In [ ]:
subprocess.run([sys.executable, "-u", str(PACKAGE/"feedback_core.py"), "--config", str(CONFIG_PATH), "--prepare-only"], check=True)
import pandas as pd
display(pd.read_csv(OUTPUT_ROOT/"run_plan.csv"))


실제 입력 음원과 교사·점수 캐시를 검증합니다. 음원 누락이나 버전 불일치는 학습 전에 알려줍니다.

In [ ]:
audit_code = "import json,sys; import feedback_core as f; c=json.load(open(sys.argv[1])); f.audit_audio(c); s,_,_=f.load_inputs(c['root']); f.verified_teacher_scores(c['root'],s)"
subprocess.run([sys.executable,"-u","-c",audit_code,str(CONFIG_PATH)],cwd=PACKAGE,check=True)


합성 파형 두 개로 모델 출력을 확인합니다. 연구 자료 학습은 다음 단계에서 시작합니다.

In [ ]:
smoke_code = """import sys,torch
import feedback_core as f
torch.set_num_threads(2)
device=torch.device(f.runtime_metadata()['device'])
for kind in ['kampa','efficientat']:
    model=f.build_model(sys.argv[1],kind).to(device).eval()
    with torch.no_grad():
        result=model(torch.randn(2,48000,device=device)*.01)
    assert result.shape==(2,9) and torch.isfinite(result).all()
    print('모델 출력 확인:',kind,device,tuple(result.shape),flush=True)
    del model
"""
subprocess.run([sys.executable,"-u","-c",smoke_code,str(PROJECT_ROOT)],cwd=PACKAGE,check=True)


## 5. 실제 실험 실행
완료된 회차는 11개 산출물의 해시를 확인한 뒤 건너뛰고, 중단된 회차는 마지막으로 완전히 저장한 에폭에서 재개합니다. 런타임이 끊기면 이 노트북을 다시 '모두 실행'하면 남은 회차부터 이어집니다. 회차마다 하는 중간 집계는 `KAMPA_결과집계.py`가 맡습니다.

In [ ]:
import datetime
log_path=OUTPUT_ROOT/("launch_"+datetime.datetime.now().strftime("%Y%m%d_%H%M%S")+".log")
command=[sys.executable,"-u",str(AGGREGATOR),"--package",str(PACKAGE),"--config",str(CONFIG_PATH)]
with log_path.open("w",encoding="utf-8") as log:
    process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in process.stdout:
            print(line,end="",flush=True);log.write(line);log.flush()
        returncode=process.wait()
    except BaseException:
        process.terminate()
        process.wait()
        raise
if returncode:
    raise RuntimeError(f"실행이 종료되었습니다(code={returncode}). 로그 확인: {log_path}")
print("계획 회차 실행 완료:",OUTPUT_ROOT)


## 6. 완료 결과 확인
계획한 모든 회차가 끝나기 전의 평균·표준편차는 중간 결과입니다.

In [ ]:
subprocess.run([sys.executable,str(AGGREGATOR),"--package",str(PACKAGE),"--aggregate",str(OUTPUT_ROOT)],check=True)
plan=pd.read_csv(OUTPUT_ROOT/"run_plan.csv")
completed=list((OUTPUT_ROOT/"runs").glob("*/complete.json"))
print(f"완료 표식 {len(completed)}개 / 계획 {len(plan)}개 (집계 시 파일 해시 검증)")
summary=OUTPUT_ROOT/"results_summary.csv"
if summary.exists():display(pd.read_csv(summary))
else:print("아직 완료된 회차가 없습니다.")
